# Getting started with ISA-PHM

Explore an ISA-PHM investigation using your own dataset. This introduction covers metadata only; it does not load measurement CSV files. Install the Wrapper in your notebook environment first (see `notebooks/README.md`).

Run the import and dataset-loading cells first. Each following example selects what it needs independently; change its study, assay, or run to explore your data.


## Imports and dataset path


In [ ]:
from pathlib import Path

from isa_phm import ISAWrapper

import pandas as pd
from IPython.display import display

# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


## Load the wrapper


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
)


## Investigation

Inspect the investigation's title, description, experiment type and study count.


In [ ]:
display(wrapper.summary())


## Contacts and publications

These tables can be empty if the information is not recorded.


In [ ]:
display(wrapper.contacts())
display(wrapper.publications())


## Studies

A dataset may have one or more studies. The example below selects the first study for the remaining exploration.


In [ ]:
import pandas as pd

studies = wrapper.list_studies()
display(pd.DataFrame([item.model_dump() for item in studies]))

if not studies:
    raise ValueError('This investigation has no studies to explore.')

study = wrapper.study(studies[0].title)
print('Selected study:', study.title)


## Study and sensors

Inspect the selected study's metadata and its sensor catalogue. Available information depends on what was recorded in your dataset.


In [ ]:
study = wrapper.study(1)

display(study.overview().model_dump())
display(study.sensor_catalog())


## Test setup images and datasheets

List the attachments referenced by the selected study, including unavailable ones. Preview every available image once, even when the same file is referenced multiple times. This is an attachment and sensor overview, not a complete structured test-setup model.


In [ ]:
import pandas as pd
study = wrapper.study(1)

attachments = wrapper.attachments(study=study.title)
display(pd.DataFrame([
    {'kind': item.kind, 'owner': item.owner, 'status': item.status,
     'path': str(item.path) if item.path else item.reference}
    for item in attachments
], columns=['kind', 'owner', 'status', 'path']))

images = {item.path: item for item in attachments
          if item.kind == 'image' and item.status == 'available' and item.path}
if images:
    for image in images.values():
        print(f"{image.owner}: {image.path.name}")
        image.display()
else:
    print('No available setup images are recorded for this study.')


## Next steps

For measurement data and analysis, choose the [diagnostic workflow](../03_workflows/02_diagnostic_workflow.ipynb) or the [prognostic workflow](../03_workflows/03_prognostic_workflow.ipynb). These examples have dataset-specific assumptions; see `notebooks/README.md`.
